# Calculating and Plotting Read Depth

First, make sure the depth has been calculated for each genomic position before continuing with this notebook. This first cell just helps confirm all of the depth files are there and that everything broadly looks okay before proceeding.

In [ ]:
import os
import pandas as pd

depth_dir = "depth_files"

# List files
depth_files = [f for f in os.listdir(depth_dir) if f.endswith(".depth.txt")]
print(f"Found {len(depth_files)} depth files")
print("Example file:", depth_files[0])

# Load the first sample and loads it into a dataframe for preview
sample_file = os.path.join(depth_dir, depth_files[0])
df = pd.read_csv(sample_file, sep='\t', header=None, names=['chrom', 'pos', 'depth'])
print(df.head())
print(df.describe())

This next cell computes the average sequencing depth in fixed (500bp) genomic windows for each sample, normalizes the coverage relative to mean nuclear depth, and then outputs normalized depth table by chromosome per sample. The output .csv files can be seen in the normalized_depth directory.

This requires the contig_conversion.csv file which is basically a conversion of the contig names generated by SeqCenter for the custom reference assembly (YL07) to the standard names of each corresponding chromosome.

In [ ]:
import os
import pandas as pd

#Directories
depth_dir = "depth_files"
output_dir = "normalized_depth"
os.makedirs(output_dir, exist_ok=True)

#Load contig-to-chromosome conversion table
conversion_path = "contig_conversion.csv"  # adjust if needed
conversion_df = pd.read_csv(conversion_path)

#Ensure consistent column naming
#Assuming conversion file columns are: 'contig' and 'chromosome'
conversion_df.rename(columns={'contig': 'chrom'}, inplace=True)

window_size = 500

def window_depth(df, window_size=500):
    df['window'] = (df['pos'] - 1) // window_size
    window_df = df.groupby(['chrom', 'window'])['depth'].mean().reset_index()
    window_df['start'] = window_df['window'] * window_size + 1
    window_df['end'] = window_df['start'] + window_size - 1
    return window_df[['chrom', 'start', 'end', 'depth']]

def normalize_depth(window_df):
    nuclear_df = window_df[~window_df['chrom'].str.contains('mt|MT|Mito', case=False, regex=True)]
    mean_depth = nuclear_df['depth'].mean()
    window_df['normalized_depth'] = window_df['depth'] / mean_depth
    return window_df, mean_depth

summary_list = []  # to collect mean depth per sample

depth_files = [f for f in os.listdir(depth_dir) if f.endswith(".depth.txt")]
print(f"Processing {len(depth_files)} samples...")

for file in depth_files:
    sample_name = file.replace(".depth.txt", "")
    print(f"Processing sample: {sample_name}")

    file_path = os.path.join(depth_dir, file)
    
    #Load depth file
    df = pd.read_csv(file_path, sep='\t', header=None, names=['chrom', 'pos', 'depth'])

    if df.empty:
        print(f"Warning: {sample_name} depth file is empty. Skipping.")
        continue

    #Compute windowed and normalized depth
    window_df = window_depth(df, window_size)
    normalized_df, mean_depth = normalize_depth(window_df)

    # nsure chrom column is string in both dataframes
    normalized_df['chrom'] = normalized_df['chrom'].astype(str)
    conversion_df['chrom'] = conversion_df['chrom'].astype(str)

    #Merge in chromosome names
    normalized_df = normalized_df.merge(conversion_df, on='chrom', how='left')

    #Save output
    output_file = os.path.join(output_dir, f"{sample_name}_normalized_depth.csv")
    normalized_df.to_csv(output_file, index=False)

    #Track mean depth
    summary_list.append({'sample': sample_name, 'mean_depth': mean_depth})

# Save mean depth summary
summary_df = pd.DataFrame(summary_list)
summary_df.to_csv(os.path.join(output_dir, "mean_depth_summary.csv"), index=False)

print("All samples processed and saved in 'normalized_depth' directory")
print("Mean depth summary saved as 'mean_depth_summary.csv'")
print("Chromosome labels added from contig_conversion.csv")

## Filtering

This code classifies each genome window in each sample as a deletion/duplication/triplication based on normalized read-depth thresholds. The reason we set the duplication threshold as 1.5 and not 2.0, for example, is that we have population-level sequencing data, so we use relaxed thresholds to capture heterogeneous CNVs expected in populations. This code also maps the corresponding genes to the CNV (if there is one). The logic is that we associate a gene with a CNV window when at least 50% of the CNV overlaps the gene. This therefore accounts for situations where a deletion likely affects a certain gene, even though it may not completely overlap with the gene. If a CNV overlaps with multiple genes, two rows will be output (which will have the same information, except the gene info column will be different).

In [ ]:
import os
import pandas as pd

#Define inputs
input_dir = "normalized_depth"
cnv_dir = "CNV_calls"
gff_file = "genome_annotation.gff3"
conversion_file = "contig_conversion.csv" #This is a file that converts the contig names to their corresponding chromosome ID
os.makedirs(cnv_dir, exist_ok=True)

#Set the CNV thresholds
dup_thresh = 1.5
trip_thresh = 2.5
del_thresh = 0.5

#Load GFF and contig conversion file
gff_cols = ['chrom', 'source', 'feature', 'start', 'end', 'score', 'strand', 'phase', 'attributes']
gff_df = pd.read_csv(gff_file, sep='\t', comment='#', names=gff_cols)
gff_genes = gff_df[gff_df['feature'] == 'gene'].copy()

#Extract gene name from attributes
def extract_gene_name(attr):
    for item in attr.split(';'):
        if item.startswith('Name='):
            return item.replace('Name=', '')
        if item.startswith('gene='):
            return item.replace('gene=', '')
    return 'unknown'
gff_genes['gene_name'] = gff_genes['attributes'].apply(extract_gene_name)

#Load contig to chromosome mapping
conv_df = pd.read_csv(conversion_file)
chrom_to_contig = dict(zip(conv_df['chromosome'], conv_df['contig']))

#Process all of the samples
norm_files = [f for f in os.listdir(input_dir) if f.endswith("_normalized_depth.csv")]
norm_files.sort()

for file in norm_files:
    sample_name = file.replace("_normalized_depth.csv", "")
    df = pd.read_csv(os.path.join(input_dir, file))

    #Assign chromosome label
    if 'chromosome' in df.columns:
        df['chrom_label'] = df['chromosome']
    else:
        df['chrom_label'] = df['chrom']

    #CNV classification
    df['CNV'] = 'neutral'
    df.loc[df['normalized_depth'] > trip_thresh, 'CNV'] = 'triplication or higher'
    df.loc[(df['normalized_depth'] > dup_thresh) & (df['normalized_depth'] <= trip_thresh), 'CNV'] = 'duplication'
    df.loc[df['normalized_depth'] < del_thresh, 'CNV'] = 'deletion'

    #Keep only non-neutral CNVs (ie. do not keep those that did not pass any of the thresholds to be called as a duplication, deletion, or triplication)
    df_cnv = df[df['CNV'] != 'neutral'].copy()
    if df_cnv.empty:
        print(f"No CNVs found for {sample_name}")
        continue
    results = []

    #Overlap logic (include all CNVs). Loop through all of the CNVs
    for _, cnv in df_cnv.iterrows():
        chrom_label = cnv['chrom_label']
        contig = chrom_to_contig.get(chrom_label, chrom_label)
        cnv_start = cnv['start']
        cnv_end = cnv['end']
        cnv_len = cnv_end - cnv_start

        #Only check the genes on the chromosome that the CNV affects
        gff_subset = gff_genes[gff_genes['chrom'] == contig]
        overlaps = []

        #Measure the overlap between the given CNV and each gene (ie. how many bp overlap between a CNV and a gene)
        for _, gene in gff_subset.iterrows():
            overlap_start = max(cnv_start, gene['start'])
            overlap_end = min(cnv_end, gene['end'])
            overlap_len = max(0, overlap_end - overlap_start)

            #Keep genes that overlap >= 50% of CNV
            if overlap_len >= 0.5 * cnv_len:
                overlaps.append({
                    'gene_name': gene['gene_name'],
                    'gene_start': gene['start'],
                    'gene_end': gene['end']
                })

        #If overlaps are found, make multiple rows. Otherwise, make one blank row
        if overlaps:
            for g in overlaps:
                results.append({
                    'chrom': cnv['chrom'],
                    'chrom_label': chrom_label,
                    'start': cnv_start,
                    'end': cnv_end,
                    'normalized_depth': cnv['normalized_depth'],
                    'CNV': cnv['CNV'],
                    'gene_name': g['gene_name'],
                    'gene_start': g['gene_start'],
                    'gene_end': g['gene_end']
                })
        else:
            results.append({
                'chrom': cnv['chrom'],
                'chrom_label': chrom_label,
                'start': cnv_start,
                'end': cnv_end,
                'normalized_depth': cnv['normalized_depth'],
                'CNV': cnv['CNV'],
                'gene_name': '',
                'gene_start': '',
                'gene_end': ''
            })

    #Save results
    out_file = os.path.join(cnv_dir, f"{sample_name}_CNV_genes.csv")
    pd.DataFrame(results).to_csv(out_file, index=False)

Next we want to filter the 40C-evolved SNVs to only keep those that are NOT found in any of the 35C-evolved populations or the fRS585 population. 

This code merges overlapping control CNV (35C-evolved or fRS585) intervals by chromosome, then subtracts those control intervals from each 40C-evolved CNV. It retains the remaining 40C-unique segment only if the total unique length is at least 50 bp. If a 40C CNV overlaps with a control CNV only partially, the overlapping part is removed and the non-overlapping remainder is kept. 

It also checks if any of the 40°C-evolved CNVs affect a gene that is also affected by one of the control populations, and counts them (even if the CNVs themselves between the 40°C-evolved and control populations don't overlap). We don't necessarily want to ignore these, but it is important to know if the gene affected is not unique to the heat-evolution.

In [ ]:
import os
import pandas as pd

cnv_dir = "CNV_calls"                 #Directory containing *_CNV_genes.csv files
output_dir = "CNV_calls_filtered"     
os.makedirs(output_dir, exist_ok=True)

control_prefixes = ["35_", "585_"]    #Control CNV files
experimental_prefix = "40_"           #Experimental CNV files
min_unique_length = 50                #Minimum bp that must be unique to 40C CNVs

#Merge overlapping intervals
def merge_intervals(intervals):
    if not intervals:
        return []
    intervals = sorted(intervals, key=lambda x: x[0])
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:
            merged[-1] = (last_start, max(end, last_end))
        else:
            merged.append((start, end))
    return merged

#Load and merge control CNVs
control_cnvs = []
control_gene_map = {}  # {chrom: {gene_name: set([pop1, pop2, ...])}}

for f in os.listdir(cnv_dir):
    if any(f.startswith(prefix) for prefix in control_prefixes) and f.endswith("_CNV_genes.csv"):
        pop_name = f.split("_CNV_genes.csv")[0]
        df = pd.read_csv(os.path.join(cnv_dir, f), dtype=str)  #Read as str to avoid float issues

        #Normalize column values early and ensure true NaNs become empty strings
        if 'chrom_label' in df.columns:
            df['chrom_label'] = df['chrom_label'].fillna("").astype(str).str.strip()
        if 'gene_name' in df.columns:
            #fillna before astype to avoid "nan" strings, then strip and remove any literal "nan"
            df['gene_name'] = df['gene_name'].fillna("").astype(str).str.strip()
            df['gene_name'] = df['gene_name'].replace("^nan$", "", regex=True)

        for chrom, group in df.groupby('chrom_label'):
            for _, row in group.iterrows():
                #Ensure start/end are numeric
                try:
                    start = int(float(row['start']))
                    end = int(float(row['end']))
                except Exception:
                    continue
                control_cnvs.append((chrom, start, end))

                gene_cell = row.get('gene_name', "")
                #Skip if empty or literal "nan"
                if not gene_cell or str(gene_cell).lower() == "nan":
                    continue
                #Split multiple gene names if present
                if (';' in gene_cell) or (',' in gene_cell):
                    parts = []
                    for sep in [';', ',']:
                        gene_cell = gene_cell.replace(sep, ';')
                    parts = [g.strip() for g in gene_cell.split(';')]
                    genes = [g for g in dict.fromkeys(parts) if g]
                else:
                    genes = [gene_cell]
                #Dedupe and ignore empty
                genes = [g for g in genes if g]
                for gene_name in genes:
                    control_gene_map.setdefault(chrom, {}).setdefault(gene_name, set()).add(pop_name)

#Merge overlapping CNVs per chromosome
control_merged = {}
if control_cnvs:
    df_ctrl = pd.DataFrame(control_cnvs, columns=['chrom', 'start', 'end'])
    for chrom, group in df_ctrl.groupby('chrom'):
        # ensure numeric
        intervals = [(int(s), int(e)) for s, e in zip(group['start'], group['end'])]
        control_merged[chrom] = merge_intervals(intervals)

#Filter experimental CNVs
exp_files = [f for f in os.listdir(cnv_dir) if f.startswith(experimental_prefix) and f.endswith("_CNV_genes.csv")]

for f in exp_files:
    df = pd.read_csv(os.path.join(cnv_dir, f), dtype=str)
    #Normalize and ensure true NaNs become empty strings
    if 'chrom_label' in df.columns:
        df['chrom_label'] = df['chrom_label'].fillna("").astype(str).str.strip()
    if 'gene_name' in df.columns:
        df['gene_name'] = df['gene_name'].fillna("").astype(str).str.strip()
        df['gene_name'] = df['gene_name'].replace("^nan$", "", regex=True)
    if 'start' in df.columns:
        df['start'] = df['start'].astype(float).astype(int)
    if 'end' in df.columns:
        df['end'] = df['end'].astype(float).astype(int)

    keep_rows = []
    for _, row in df.iterrows():
        chrom = row['chrom_label']
        cnv_start, cnv_end = int(row['start']), int(row['end'])
        cnv_len = cnv_end - cnv_start
        control_intervals = control_merged.get(chrom, [])

        #Count how many control populations this gene appears in
        gene_cell = row.get('gene_name', "")
        gene_overlap_count = 0
        #Skip counting when gene_cell is empty or literal "nan"
        if gene_cell and str(gene_cell).lower() != "nan":
            if (';' in gene_cell) or (',' in gene_cell):
                for sep in [';', ',']:
                    gene_cell = gene_cell.replace(sep, ';')
                genes = [g.strip() for g in gene_cell.split(';') if g.strip()]
            else:
                genes = [gene_cell]
            pops = set()
            for g in genes:
                pops.update(control_gene_map.get(chrom, {}).get(g, set()))
            gene_overlap_count = len(pops)

        #Compute unique regions
        unique_regions = [(cnv_start, cnv_end)]
        partially_overlaps = False
        for c_start, c_end in control_intervals:
            temp = []
            for u_start, u_end in unique_regions:
                # no overlap
                if u_end <= c_start or u_start >= c_end:
                    temp.append((u_start, u_end))
                else:
                    partially_overlaps = True
                    #Overlap: keep the left portion
                    if u_start < c_start:
                        temp.append((u_start, c_start))
                    #Keep the right portion
                    if u_end > c_end:
                        temp.append((c_end, u_end))
            unique_regions = temp
        unique_length = sum(u_end - u_start for u_start, u_end in unique_regions)

        if unique_length >= min_unique_length:
            for u_start, u_end in unique_regions:
                new_row = row.copy()
                new_row['start'] = u_start
                new_row['end'] = u_end
                new_row['gene_overlap_in_controls_count'] = gene_overlap_count
                new_row['partially_overlaps_control'] = partially_overlaps
                keep_rows.append(new_row)

    if not keep_rows:
        print(f"No unique CNVs found for {f}")
        continue

    out_df = pd.DataFrame(keep_rows)

    #Reorder columns: chrom_label first, then metadata
    if 'chrom_label' in out_df.columns:
        cols = ['chrom_label'] + [c for c in out_df.columns if c != 'chrom_label']
        out_df = out_df[cols]

    #Write output
    out_file = os.path.join(output_dir, f"{f.replace('_CNV_genes.csv','')}_unique_to_40C.csv")
    out_df.to_csv(out_file, index=False)

    #Safely handle NaN chromosome labels
    chrom_list = ", ".join(
        str(chrom) for chrom in out_df['chrom_label'].dropna().unique()
    )
    if chrom_list:
        print(f"   Chromosomes involved: {chrom_list}\n")
    else:
        print("   No valid chromosome labels (likely unmapped contigs)\n")

This next script will filter the 40°C-unique CNV files for ONLY the genic CNVs that were associated with a gene that was not identified in any non-40°C-evolved population:

In [ ]:
import os
import pandas as pd

#Inputs
input_dir = "CNV_calls_filtered"
output_dir = "genic_CNV_calls_filtered"      
os.makedirs(output_dir, exist_ok=True)

#Iterate through 40C CNV files
files = [f for f in os.listdir(input_dir) if f.endswith("_unique_to_40C.csv")]
if not files:
    raise ValueError("No *_unique_to_40C.csv files found in CNV_calls_filtered!")

for f in files:
    in_path = os.path.join(input_dir, f)
    df = pd.read_csv(in_path)

    #Skip empty files
    if df.empty:
        print(f"Skipping {f} (empty file)")
        continue

    #Filter (must have a valid gene name)
    df = df[df['gene_name'].notna() & (df['gene_name'].astype(str).str.strip() != '')]

    #Filter (gene_overlap_in_controls_count must be 0 or NaN)
    if 'gene_overlap_in_controls_count' in df.columns:
        df['gene_overlap_in_controls_count'] = pd.to_numeric(df['gene_overlap_in_controls_count'], errors='coerce')
        df = df[df['gene_overlap_in_controls_count'].fillna(0) == 0]
    else:
        print(f"Column 'gene_overlap_in_controls_count' not found in {f}. Keeping all remaining rows.")

    #Save output if anything remains
    if not df.empty:
        out_path = os.path.join(output_dir, f)
        df.to_csv(out_path, index=False)
        print(f"{len(df)} genic CNVs saved to {out_path}")
    else:
        print(f"No genic CNVs passed filters in {f}")

The next script will FURTHER take those and collapse the rows that share a gene into one row (ie. one row per gene with the corresponding CNV info summarized):

In [ ]:
import os
import pandas as pd
import numpy as np

#Inputs
input_dir = "genic_CNV_calls_filtered"
output_dir = os.path.join(input_dir, "merged_genic_CNV_calls_filtered")
os.makedirs(output_dir, exist_ok=True)

#Helper: merge function for groupby
def merge_values(series):
    #Merge unique non-null values as comma-separated strings
    vals = series.dropna().astype(str).unique()
    return ", ".join(sorted(vals)) if len(vals) > 0 else np.nan

#Iterate through all filtered CNV files
files = [f for f in os.listdir(input_dir) if f.endswith("_unique_to_40C.csv")]
if not files:
    raise ValueError(f"No *_unique_to_40C.csv files found in {input_dir}!")

for f in files:
    in_path = os.path.join(input_dir, f)
    df = pd.read_csv(in_path)

    if df.empty:
        print(f"Skipping {f} (empty file)")
        continue

    if "gene_name" not in df.columns:
        print(f"No 'gene_name' column in {f}, skipping.")
        continue

    #Prepare normalized_depth as numeric
    if "normalized_depth" in df.columns:
        df["normalized_depth"] = pd.to_numeric(df["normalized_depth"], errors="coerce")

    #Define aggregation rules
    agg_dict = {}
    for col in df.columns:
        if col == "gene_name":
            continue
        elif col in ["start", "gene_start"]:
            agg_dict[col] = "min"
        elif col in ["end", "gene_end"]:
            agg_dict[col] = "max"
        elif col == "normalized_depth":
            # keep all as comma-separated string for reference
            agg_dict[col] = merge_values
        else:
            agg_dict[col] = merge_values

    #Group by gene_name and aggregate
    grouped = df.groupby("gene_name", dropna=False).agg(agg_dict).reset_index()

    #Compute mean normalized_depth per gene
    if "normalized_depth" in df.columns:
        def mean_depth(val_str):
            if pd.isna(val_str):
                return np.nan
            vals = [float(x) for x in val_str.split(",") if x.strip() != ""]
            return np.mean(vals) if vals else np.nan
        grouped["mean_normalized_depth"] = grouped["normalized_depth"].apply(mean_depth)

    #Save output
    out_path = os.path.join(output_dir, f)
    grouped.to_csv(out_path, index=False)
    print(f"{len(grouped)} merged gene-level CNVs saved to {out_path}")

print("Gene-collapsed CNV files saved in 'merged_genic_CNV_calls_filtered/'")

This final code just merges all of the individual files into one master table.

In [ ]:
import os
import pandas as pd
import numpy as np

#Inputs
input_dir = "genic_CNV_calls_filtered/merged_genic_CNV_calls_filtered"
output_dir = "genic_CNV_calls_filtered/merged_genic_CNV_calls_filtered"
os.makedirs(output_dir, exist_ok=True)
combined_output_file = os.path.join(output_dir, "all_40C_genic_CNVs_merged.csv")

#Helper to merge values
def merge_values(series):
    vals = series.dropna().astype(str).unique()
    return ", ".join(sorted(vals)) if len(vals) > 0 else np.nan

#List of merged CNV files
files = [f for f in os.listdir(input_dir) if f.endswith("_unique_to_40C.csv")]
if not files:
    raise ValueError(f"No *_unique_to_40C.csv files found in {input_dir}!")

all_dfs = []

for f in files:
    in_path = os.path.join(input_dir, f)
    df = pd.read_csv(in_path)

    if df.empty or "gene_name" not in df.columns:
        print(f"Skipping {f} (empty or missing 'gene_name')")
        continue

    #Extract sample name from filename (before first '_unique_to_40C.csv')
    sample_name = f.replace("_unique_to_40C.csv", "")
    df["sample"] = sample_name

    #Define aggregation rules for gene-level collapse
    agg_dict = {}
    for col in df.columns:
        if col == "gene_name" or col == "sample":
            continue
        elif col in ["start", "gene_start"]:
            agg_dict[col] = "min"
        elif col in ["end", "gene_end"]:
            agg_dict[col] = "max"
        elif col == "normalized_depth":
            agg_dict[col] = merge_values
        else:
            agg_dict[col] = merge_values

    #Group by gene_name and sample
    grouped = df.groupby(["gene_name", "sample"], dropna=False).agg(agg_dict).reset_index()

    #Compute mean normalized_depth per gene
    if "normalized_depth" in df.columns:
        def mean_depth(val_str):
            if pd.isna(val_str):
                return np.nan
            vals = [float(x) for x in val_str.split(",") if x.strip() != ""]
            return np.mean(vals) if vals else np.nan
        grouped["mean_normalized_depth"] = grouped["normalized_depth"].apply(mean_depth)

    all_dfs.append(grouped)

#Concatenate all samples into one DataFrame
combined_df = pd.concat(all_dfs, ignore_index=True)

#Save combined CSV
combined_df.to_csv(combined_output_file, index=False)
print(f"Combined merged CNV file saved: {combined_output_file}")
print(f"   Total rows: {len(combined_df)}, samples included: {combined_df['sample'].nunique()}")